# Introduction to Retrieval-Augmented Generation (RAG)

RAG combines retrieval systems with generative models to improve how NLP tasks are handled. Decoder-only models like ChatGPT generate text purely from patterns learned during training, which makes them prone to factual errors and gaps on domain-specific or rare knowledge — they have no way to look anything up. RAG fixes this by adding a retrieval step: before generating a response, the model dynamically pulls relevant information from an external knowledge source and uses it as context, rather than relying solely on what it memorized during training.

The architecture (fig below) breaks this into distinct stages: **Query Processing** prepares the user's question, the **Knowledge Base** and its **Document Embeddings** are searched by the **Retrieval System**, the **Document Retriever** pulls back the matching material as **Context**, that context is folded into a **Prompt**, and the **Decoder Model** uses it to produce the final **Generated Response**.

## What we're going to do

This notebook builds that pipeline end to end — a baseline RAG system implementing each stage in the diagram, as the starting point before layering on the more advanced techniques (hybrid/keyword search, query transformations, corrective retrieval) explored in the other notebooks in this repo.

![RAG Architecture](../images/ch06__image001.png)

## Why We Need Chunking

Embedding models turn a piece of text into one fixed-size vector by pooling across every token they read. That creates two independent problems if you try to embed a whole document as a single unit instead of splitting it up first:

1. **Context window overflow.** Every embedding model has a fixed maximum input length (e.g. `nomic-embed-text-v1.5` caps at 8,192 tokens, ~6,000 words). Feed it something longer and it doesn't error — it silently **truncates**, discarding everything past the limit before the model ever sees it. If the answer to a future query lives past that cutoff, the embedding was never computed from it, so no similarity search will ever surface that document for that query — not because retrieval failed, but because the information was thrown away before embedding happened.
2. **Semantic dilution.** Even when a document *does* fit, cramming many unrelated ideas into one vector blends them together via pooling. A chunk covering three different topics produces a vector that's an average of all three — "close" to none of them in vector space. A query about any single topic scores lower against this diluted vector than it would against a chunk that talked about only that topic. Chunking keeps each vector's "topic" narrow enough for similarity search to actually tell chunks apart.

The tradeoff: chunk too small (single sentences) and you lose the surrounding context a reader needs to make sense of the fragment. Chunk too large and you're back to dilution. There's no universal right size — it's tuned per corpus, which is what `5. Indexing` explores directly (granular vs. coarse chunking).

## Recursive Chunking

The naive way to chunk is **fixed-size splitting**: cut the text every N characters (with some overlap). It's simple, but it cuts blindly — it will happily slice a sentence in half if the boundary lands mid-word, destroying the exact local coherence chunking is supposed to preserve.

**Recursive chunking** fixes this by splitting on a *prioritized list of separators*, from "biggest structural break" down to "smallest," and only falling back to a smaller separator when the current one isn't enough to get chunks under the target size. A typical separator hierarchy:

1. `"\n\n"` — paragraph breaks (try this first; it respects the author's own structure)
2. `"\n"` — line breaks
3. `". "` — sentence boundaries
4. `" "` — word boundaries
5. `""` — raw characters (last resort, only if nothing above worked)

**The "recursive" part:** after splitting on separator 1, any resulting piece that's *still* too large gets split again — on separator 2, then 3, and so on — recursively, until every piece fits under the target chunk size. A short paragraph might need no further splitting at all; a long one might recurse all the way down to sentence-level. This is why the technique keeps as much natural structure as possible (whole paragraphs where they fit) while still guaranteeing a hard size limit everywhere else.

In practice, this is exactly what LangChain's `RecursiveCharacterTextSplitter` does — and `langchain-text-splitters` is already a dependency in this repo's `requirements.txt`:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # target max characters per chunk
    chunk_overlap=50,     # characters repeated between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],  # tried in this order
)

chunks = splitter.split_text(long_document_text)
```

Compare this to `5. Indexing`'s `TokenTextSplitter`, which chunks by token count rather than character count and separator hierarchy — a good next read once this section makes sense

## Imports

Every reusable function used across the notebooks in this folder — chunking, embeddings, Qdrant indexing/retrieval, generation, and evaluation — lives in this folder's `utils.py`, imported once here at the top rather than defined inline where each concept is introduced.

In [ ]:
from utils import (
    recursive_split_documents,
    get_text_embeddings,
    index_documents,
    generate_answer,
    build_synthetic_eval_set,
    evaluate_retrieval,
    evaluate_generation,
)

## Loading the Source Documents

Before anything can be chunked, embedded, or retrieved, we need a corpus. This notebook uses a Kaggle dataset of research paper abstracts (`dblp-v10`), pulled via `kagglehub`.

Each row becomes a dict with two parts: `page_content` (the abstract itself — the text we'll actually chunk and embed) and `metadata` (title, authors, year, venue, and paper id — information we want to keep attached to the text for citations later, but don't want mixed into the embedding). This `page_content`/`metadata` split is exactly the shape LangChain's document utilities expect as separate `texts` and `metadatas` lists in the next step.

In [ ]:
import os

import pandas as pd
import kagglehub

kagglehub.login()

path = kagglehub.dataset_download("nechbamohammed/research-papers-dataset")
df = pd.read_csv(os.path.join(path, "dblp-v10.csv"))

df = df[:500].dropna(subset=['abstract']).copy()

data = []
for row_num, row in df.iterrows():
    if row['abstract'] != 'NaN':
        data.append({
            "page_content": row['abstract'],
            "metadata": {
                "source": row["title"],
                "authors": row["authors"],
                "year": row["year"],
                "venue": row["venue"],
                "paper_id": row["id"]
            }
        })

print(f'You have {len(data)} document(s) in your data')

## Applying Recursive Chunking to Our Documents

Now that we have raw abstracts as `page_content`/`metadata` dicts, we apply the `RecursiveCharacterTextSplitter` described above. Abstracts are short and already single-topic, so most won't need splitting at all — a larger `chunk_size` (1024 characters, `recursive_split_documents`'s default) is used here than in the illustrative example earlier, since we're not fighting dilution the way we would with full paper bodies.

`splitter.create_documents(...)` propagates each source document's `metadata` onto every chunk it produces, so a paper split into three chunks still has all three traceable back to the same `paper_id`, `title`, and `authors` — this is what lets us cite sources correctly once retrieval and generation are wired up.

In [ ]:
splits = recursive_split_documents(data[:100])
print(f"You have {len(splits)} chunk(s) from {len(data)} document(s)")

## Choosing an Embedding Model

To search chunks by meaning rather than exact keywords, each one needs to become a vector. We use `nomic-embed-text-v1.5` loaded directly through Hugging Face `transformers` (tokenizer + model), rather than calling an embeddings API — this keeps the pipeline free of per-call cost and able to run fully offline once the model weights are cached locally.

`trust_remote_code=True` is required because Nomic ships custom modeling code alongside the weights rather than relying only on a stock `transformers` architecture.

This model is loaded here, in the notebook, rather than inside `utils.py` — `get_text_embeddings` takes the tokenizer and model as explicit arguments instead of loading its own copy, so importing `utils.py` doesn't force every notebook in this folder to eagerly download a model it might not need (the hotel-review notebook uses a different embedding setup entirely).

In [ ]:
from transformers import AutoTokenizer, AutoModel

text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

## Generating Embeddings for Each Chunk

`get_text_embeddings` (in `utils.py`) tokenizes a chunk, runs it through the model with gradient tracking disabled (we're not training anything, so this saves memory), and mean-pools `last_hidden_state` across every token to collapse the sequence into one fixed-size vector — the exact pooling operation discussed in "Why We Need Chunking" above, and the reason chunk size directly affects embedding quality.

Applying this to every chunk in `splits` produces `text_embedded`: a list of vectors aligned 1:1 with `splits`, ready to be written into a vector database.

In [ ]:
text_embedded = [get_text_embeddings(document.page_content, text_tokenizer, text_model) for document in splits]

## Setting Up a Vector Database

With embeddings in hand, we need somewhere to store them for fast similarity search — this is Qdrant's job. `QdrantClient(":memory:")` runs Qdrant in-process for this notebook: no server to stand up, but also no persistence — the index disappears the moment the kernel restarts. Swap `":memory:"` for a file path or a running Qdrant server URL once this moves past prototyping.

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

## Indexing Chunks into Qdrant

`index_documents` first creates a collection sized to match the embedding dimensionality, configured for cosine distance — the standard choice for text embeddings, since it measures the angle between vectors (their direction/meaning) rather than magnitude.

It then uploads every chunk as a "point": a random UUID as its id, its embedding vector, and a payload carrying the original text and metadata. That payload is what we get back from a search — it's how retrieval can hand the generation step actual text and citations, not just a bare vector.

In [ ]:
index_documents(client, "research_collection", splits, text_embedded)

## Retrieving Relevant Chunks

This is the retrieval step of RAG. `query_qdrant` (in `utils.py`) embeds the user's query with the *same* embedding model used for the documents — this is non-negotiable: query and document vectors must come from the same embedding space to be comparable at all — then asks Qdrant for the nearest points by cosine similarity.

Each hit's payload (content + metadata) is unpacked into a plain result dict. It's used directly inside `generate_answer` below, rather than called on its own here.

## Generating the Final Answer

This is the generation step. Retrieved chunks become the "Context" folded into a prompt that instructs the model to answer using citations back to the source papers — the same architecture sketched in the intro at the top of this notebook.

`generate_answer` (in `utils.py`) reuses the Claude client already configured in `llm_model.py` at the repo root, rather than standing up a separate model, and streams tokens as they're generated so a reader watching the notebook run sees the answer appear incrementally instead of waiting on the full response.

## Running the Pipeline End to End

With chunking, embedding, indexing, retrieval, and generation all wired up, we can run a real question through the whole thing — from raw text to a cited, Markdown-formatted answer grounded in the papers we indexed.

In [ ]:
query = "an autoassociative neural network with dynamic synapses"

response, sources = generate_answer(query, client, text_tokenizer, text_model)

## Evaluating the RAG Pipeline

A good final answer can hide a broken middle step — the LLM is skilled enough to sound confident even when it was handed the wrong context, or no context at all. To actually trust this pipeline, each stage needs to be checked independently:

- **Chunking** has no direct metric of its own. It's evaluated indirectly, through its effect on retrieval — sweep `chunk_size` and watch whether retrieval quality improves or degrades.
- **Retrieval** needs a labeled set of "this question is answered by that chunk" pairs, so we can check whether the right chunk actually comes back in the top-k results.
- **Generation** needs to be checked against the context it was actually given, not against vibes — did it stick to what the retrieved chunks support, and did it address the question that was asked?

We don't have human-labeled question/chunk pairs for this dataset, so the next section builds a synthetic evaluation set with the LLM itself, then uses it to score retrieval, chunking, and generation in turn.

## Building a Synthetic Evaluation Set

For a sample of chunks, we ask the LLM to write one specific question that chunk answers (`build_synthetic_eval_set`, in `utils.py`). Each `(question, source_chunk)` pair becomes a ground-truth label: retrieval "succeeds" on that question if the source chunk's paper shows up in the top-k results, and generation is later checked against the same source.

We key ground truth on `paper_id` (from each chunk's metadata) rather than the exact chunk text. That matters once chunk boundaries shift (e.g. a different `chunk_size`) — the original chunk's exact text may no longer exist anywhere in a re-chunked index, but the paper it came from still does.

This isn't as rigorous as human-labeled questions, but it's enough to catch a badly tuned chunk size or a broken embedding/retrieval step.

In [ ]:
eval_set = build_synthetic_eval_set(splits, n=20)
print(f"Built {len(eval_set)} synthetic eval questions")

## Evaluating Retrieval

For each synthetic question, `evaluate_retrieval` (in `utils.py`) runs it through `query_qdrant` and checks whether any of the top-k results traces back to the same `paper_id` the question was generated from.

- **Recall@k** — the fraction of questions where the correct paper appears anywhere in the top-k results. Since the source chunk is guaranteed to already be indexed, a low score here means the embedding model or similarity search is failing to place semantically matching text near each other in vector space — not that the information doesn't exist.

$$\text{Recall@k} = \frac{1}{|Q|} \sum_{q \in Q} \mathbb{1}\left[\text{rank}_q \le k\right]$$

where $Q$ is the set of evaluation questions and $\text{rank}_q$ is the position of the correct paper in question $q$'s results (or $\infty$ if it never appears).

- **MRR (Mean Reciprocal Rank)** — rewards ranking a correct hit near the *top* of the results, not just anywhere in the top-k. A system with high recall but low MRR is finding the right chunk, but burying it under less-relevant ones.

$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}$$

with the convention $\frac{1}{\text{rank}_q} = 0$ when the correct paper doesn't appear in the top-k at all.

In [ ]:
retrieval_scores = evaluate_retrieval(eval_set, client, text_tokenizer, text_model, k=5)
print(retrieval_scores)

## Evaluating Generation with an LLM Judge

Generation is graded on two axes that don't require a ground-truth answer to write:

- **Faithfulness** — does every claim in the generated answer actually trace back to the retrieved context, or is the model adding facts it wasn't given?
- **Relevancy** — does the answer actually address the question that was asked, rather than just reciting the context?

The standard tool for this is `ragas`, but its LLM wrapper still imports `ChatVertexAI` from a `langchain_community` module that no longer exists in the `langchain-community` version this repo runs (it requires `langchain-community<0.4`, which in turn requires `langchain<1.0.0` — a downgrade that would break every other notebook in this repo built on LangChain's v1 API). Rather than downgrade `langchain` repo-wide for one eval cell, `utils.py` implements the same idea directly (`judge_answer`, `parse_llm_json`, `evaluate_generation`): ask the LLM itself (the same `llm` from `llm_model.py` used for generation) to score each `(question, context, answer)` triple against a fixed rubric, and average the scores. This is exactly what `ragas`'s metrics do under the hood — an LLM-as-judge — just without the extra dependency.

In [ ]:
generation_scores = evaluate_generation(eval_set, client, text_tokenizer, text_model, n=10)
print(generation_scores)

## Building a Held-Out Test Set

The evaluation above has a flaw worth naming: `eval_set` was built by asking the LLM to write questions about the *same 100 documents* (`data[:100]`) that were indexed. Even keying on `paper_id` instead of exact chunk text doesn't fix this — the content those questions are about was fully present in the index the whole time. A retrieval system could look far better here than it actually is.

A proper held-out test uses content the pipeline has never seen: index a second, previously-unused batch of 100 documents (`data[100:200]`) into the *same* collection, then generate the evaluation questions exclusively from that new batch. Retrieval now has to find the right chunk among a larger, more realistic index, and the questions are about documents that played no part in building or tuning anything above.

In [ ]:
held_out_documents = data[100:200]

held_out_splits = recursive_split_documents(held_out_documents)
held_out_embeddings = [get_text_embeddings(doc.page_content, text_tokenizer, text_model) for doc in held_out_splits]

index_documents(client, "research_collection", held_out_splits, held_out_embeddings)
print(f"Indexed {len(held_out_splits)} additional held-out chunk(s) from {len(held_out_documents)} document(s)")

## Running the Held-Out Evaluation

With the held-out batch indexed, build questions from it exclusively and rerun the same retrieval and generation metrics used above. The index now also contains the original 100 documents as distractors, so a correct retrieval has to outrank them, not just outrank an empty or trivial search space — the closest thing to a real "unseen queries against a live index" test this notebook can do without external human-labeled data.

In [ ]:
held_out_eval_set = build_synthetic_eval_set(held_out_splits, n=20)
print(f"Built {len(held_out_eval_set)} held-out eval question(s)")

held_out_retrieval_scores = evaluate_retrieval(held_out_eval_set, client, text_tokenizer, text_model, k=5)
print("Held-out retrieval:", held_out_retrieval_scores)

held_out_generation_scores = evaluate_generation(held_out_eval_set, client, text_tokenizer, text_model, n=10)
print("Held-out generation:", held_out_generation_scores)